[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part1_foundation/ch03_llm_api.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 3장 LLM API 프로그래밍
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제1부 토대 — LLM과 프롬프트
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
!pip install -q google-genai

### API 키와 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.5-flash"   # 모델 갱신 시 이 한 줄만 변경

### 미니 챗봇 헬퍼

In [ ]:
# 미니 챗봇용 상수·헬퍼 (신 SDK 대화 형식)
SYSTEM_MESSAGE = {"role": "user", "parts": [{"text": "너는 간결하고 친절한 한국어 도우미다."}]}
def make_user(t):  return {"role": "user",  "parts": [{"text": t}]}
def make_model(t): return {"role": "model", "parts": [{"text": t}]}
PROMPT_MARK = "나> "
EXIT_WORD = "끝"

## 예제 3-1. 가장 단순한 첫 API 호출

In [ ]:
from google import genai

client = genai.Client(api_key=API_KEY)
res = client.models.generate_content(
    model=GEMINI_FLASH,
    contents=PROMPT,
)
print(res.text)

## 예제 3-2. 온도에 따른 응답 비교

In [ ]:
prompt = "여름 신메뉴 아이스크림 이름을 세 개만 제안해줘"
for temperature in [0.0, 1.0]:
    res = client.models.generate_content(
        model=GEMINI_FLASH,
        contents=prompt,
        config={"temperature": temperature},
    )
    print(f"[온도 {temperature}]\n{res.text}")

## 예제 3-3. 스키마 기반 구조화 출력

In [ ]:
import json

schema = {
    "type": "object",
    "properties": {
        "분류": {"type": "string"},
        "확신도": {"type": "number"},
    },
    "required": ["분류", "확신도"],
}

res = client.models.generate_content(
    model=GEMINI_FLASH,
    contents="다음 문의를 분류해 줘: 옷을 다시 돌려주세요",
    config={
        "response_mime_type": "application/json",
        "response_schema": schema,
        "temperature": 0,
    },
)
print(json.loads(res.text))

## 예제 3-4. 함수 호출로 도구 쓰기

In [ ]:
from google.genai import types

weather_function = types.FunctionDeclaration(
    name="get_weather",
    description="도시 이름을 받아 현재 날씨를 조회한다.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
        },
        "required": ["city"],
    },
)
tools = types.Tool(function_declarations=[weather_function])

res = client.models.generate_content(
    model=GEMINI_FLASH,
    contents="오늘 서울 날씨 알려줘",
    config=types.GenerateContentConfig(tools=[tools]),
)

call = res.candidates[0].content.parts[0].function_call
print(call.name, call.id, dict(call.args))

## 예제 3-5. 토큰 사용량 확인

In [ ]:
res = client.models.generate_content(
    model=GEMINI_FLASH,
    contents="LLM을 한 문장으로 설명해 줘",
)
usage = res.usage_metadata
print("입력 토큰:", usage.prompt_token_count)
print("출력 토큰:", usage.candidates_token_count)
print("합계:", usage.total_token_count)

## 예제 3-6. 요소를 묶은 미니 챗봇

In [ ]:
history = [SYSTEM_MESSAGE]
while True:
    user = input(PROMPT_MARK)
    if user == EXIT_WORD:
        break
    history.append(make_user(user))
    reply = client.models.generate_content(
        model=GEMINI_FLASH,
        contents=history,
    ).text
    print(reply)
    history.append(make_model(reply))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 3장 노트북